# Step 2 on a Kaggle GPU: rest of the tuning, final runs, DCRNN-sized model

Runs the same code as on the PC (`src/utils/tuning/nn_tuning.py`), only on a GPU:
1. finish the LSTM tuning (20 tries) and tune the encoder-decoder LSTM (20 tries), on the validation set;
2. final runs with the best settings: Linear, TCN, LSTM, encoder-decoder LSTM, seeds 42 / 1 / 2, each evaluated once on the test set;
3. the encoder-decoder LSTM with the size of FC-LSTM in the DCRNN paper (2 layers x 256 units), seed 42;
4. everything needed back on the PC goes into `results.zip` (Output tab).

**Setup on Kaggle:** Add Input -> the dataset with `kaggle_upload.zip`; Settings -> Accelerator: **GPU T4 x2**, Internet: **on**.
Then **Save Version -> Save & Run All (Commit)**: it runs in the background, the browser can be closed.

In [ ]:
import glob, json, os, shutil, subprocess, sys, time, zipfile

# the uploaded code + data + tuning database, wherever Kaggle put it; /kaggle/input is read-only, so work on a copy
WORK = '/kaggle/working/topic4'
marker = glob.glob('/kaggle/input/**/src/utils/tuning/nn_tuning.py', recursive=True)
if marker:                                   # Kaggle unpacked the zip
    shutil.copytree(marker[0].removesuffix('/src/utils/tuning/nn_tuning.py'), WORK, dirs_exist_ok=True)
else:                                        # the zip was kept as a file: unpack it here
    zipfile.ZipFile(glob.glob('/kaggle/input/**/*.zip', recursive=True)[0]).extractall('/kaggle/working')
os.chdir(WORK)
print(sorted(os.listdir('.')), '| tuning database found:', os.path.exists('checkpoints/tuning/optuna.db'))

In [ ]:
# BasicTS without its pinned extras (its old `transformers` does not build on Kaggle and our code does not use it),
# plus the few small packages our code imports that Kaggle does not have
!pip install -q --no-deps basicts==1.1.0 easy-torch==1.3.3
!pip install -q easydict setproctitle optuna
!python -c "import basicts, easytorch, optuna; from src.utils.tuning.nn_tuning import MODELS; print('BasicTS', basicts.__version__, '| models:', list(MODELS))"
import torch
assert torch.cuda.is_available(), 'Settings -> Accelerator -> GPU T4 x2'
print(torch.__version__, torch.cuda.device_count(), 'x', torch.cuda.get_device_name(0))

## 1. Tuning (continues the studies from the PC; tries already done are kept)

In [ ]:
!python -m src.utils.tuning.nn_tuning tune --models LSTM EncoderDecoderLSTM --trials 20 --workers 3

## 2–3. Final runs and the DCRNN-sized model, all at the same time (logs: `checkpoints/final/*.log`)

BasicTS trains on "GPU 0" of what a process can see, so each process is shown only one of the two GPUs; the two heavy jobs (encoder-decoder LSTM, DCRNN size) run on different GPUs.

In [ ]:
os.makedirs('checkpoints/final', exist_ok=True)
FINAL = [sys.executable, '-m', 'src.utils.tuning.nn_tuning', 'final', '--threads', '1']
jobs = [  # (name, GPU, arguments)
    ('Linear_TCN', 0, ['--models', 'Linear', 'TCN', '--seeds', '42', '1', '2']),
    ('LSTM', 1, ['--models', 'LSTM', '--seeds', '42', '1', '2']),
    ('EncoderDecoderLSTM', 0, ['--models', 'EncoderDecoderLSTM', '--seeds', '42', '1', '2']),
    ('DCRNN_size', 1, ['--models', 'EncoderDecoderLSTM', '--seeds', '42', '--tag', 'dcrnn_size',
                       '--params', '{"hidden_size": 256, "num_layers": 2, "dropout": 0.0}'])]
n_gpus = torch.cuda.device_count()
procs = {}
for name, gpu, args in jobs:
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu % n_gpus))     # this process sees only this GPU
    procs[name] = subprocess.Popen(FINAL + args, env=env, stderr=subprocess.STDOUT,
                                   stdout=open(f'checkpoints/final/{name}.log', 'w'))
time.sleep(180)                                                        # then show which GPU runs what
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
print({name: p.wait() for name, p in procs.items()})                  # exit code 0 = finished without errors

## 4. Results and download

In [ ]:
!python -m src.utils.tuning.nn_tuning summary

In [ ]:
import pandas as pd
rows = []
for path in sorted(glob.glob('checkpoints/final/*/*/*/test_metrics.json')):
    model, run = path.split('/')[2:4]
    m = json.load(open(path))
    rows.append({'model': model, 'run': run,
                 **{f'test MAE {mins} min': m[f'horizon_{h}']['MAE'] for h, mins in [(3, 15), (6, 30), (12, 60)]}})
pd.DataFrame(rows).round(3)

In [ ]:
def keep(path):        # small files of every run + the predictions of the seed-42 runs (for the error analysis)
    if path.endswith(('test_metrics.json', 'cfg.json', '.log', '.csv', '.db', '_best_val_MAE.pt')):
        return True
    return '_seed42' in path and path.endswith(('prediction.npy', 'targets.npy'))

with zipfile.ZipFile('/kaggle/working/results.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for path in glob.glob('checkpoints/tuning/**', recursive=True) + glob.glob('checkpoints/final/**', recursive=True):
        if os.path.isfile(path) and keep(path):
            z.write(path)
print(f"results.zip: {os.path.getsize('/kaggle/working/results.zip') / 1e6:.0f} MB -> Output tab -> download, "
      "then unzip it in the repo folder on the PC")